# BodyMaps | CT AI Lab — Final Colab Demo

**Research-only prototype. Not for clinical decision-making.**

This notebook reproduces the verified browser-triggered SuPreM CT segmentation demonstration used for the BodyMaps developer project.

It performs one clean workflow:

1. create the known-working isolated Python 3.9 SuPreM environment;
2. pin the SuPreM source to a known commit;
3. download and SHA-256 verify the official pretrained U-Net checkpoint;
4. replace the upstream positional checkpoint assignment with a strict name-based loader in a **separate copy** of the inference script;
5. run a checkpoint semantic audit;
6. install the Streamlit web layer;
7. fetch the reviewed `app.py` and `inference_backend.py` from the public project repository at a pinned commit;
8. launch Streamlit on the Colab GPU host;
9. expose the private development session with a temporary Cloudflare Quick Tunnel.

The browser is then the inference test: upload an **anonymized** CT ZIP, click **Run pretrained SuPreM inference**, and verify that nine organ masks are generated.

**No CT scans, reference masks, model checkpoint, or inference logs are committed to GitHub.**

## 1. GPU and isolated SuPreM environment

Select a **T4 or A100 GPU** in Colab before running the notebook.

The Colab kernel itself may use a newer Python version; SuPreM runs inside its own Python 3.9 environment at `/content/suprem_env`.

In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil, json, hashlib, time, re, socket, urllib.request

ROOT = Path("/content")
REPO = ROOT / "SuPreM"
SRC = REPO / "direct_inference"
VENV_DIR = ROOT / "suprem_env"
VENV = VENV_DIR / "bin" / "python"
ENV_MARKER = VENV_DIR / ".bodymaps_final_ready"
REPORT = ROOT / "bodymaps_reports"
WEBAPP = ROOT / "bodymaps_webapp"

REPORT.mkdir(parents=True, exist_ok=True)
WEBAPP.mkdir(parents=True, exist_ok=True)

RUN_ENV = os.environ.copy()
RUN_ENV.update(MPLBACKEND="Agg", PYTHONUNBUFFERED="1")

def run(cmd, cwd=None, show_output=True):
    cmd = [str(x) for x in cmd]
    print("RUN:", " ".join(cmd), flush=True)
    p = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=RUN_ENV,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if show_output and p.stdout:
        print(p.stdout, end="" if p.stdout.endswith("\n") else "\n")
    if p.returncode != 0:
        if not show_output and p.stdout:
            print(p.stdout[-12000:])
        raise RuntimeError(f"Command failed (exit {p.returncode}): {cmd[0]}")
    return p

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("GPU not visible. In Colab choose Runtime → Change runtime type → T4/A100.")

run(["nvidia-smi"])
print("Colab kernel Python:", sys.version.split()[0])

run([sys.executable, "-m", "pip", "install", "-q", "uv"])
UV = shutil.which("uv")
if not UV:
    raise RuntimeError("uv command not found after installation.")

run([UV, "python", "install", "3.9"])

valid_existing = VENV.is_file() and ENV_MARKER.is_file()
if valid_existing:
    version = subprocess.check_output([str(VENV), "--version"], text=True).strip()
    valid_existing = version.startswith("Python 3.9.")

if not valid_existing:
    if VENV_DIR.exists():
        shutil.rmtree(VENV_DIR)
    run([UV, "venv", "--python", "3.9", str(VENV_DIR)])

version = subprocess.check_output([str(VENV), "--version"], text=True).strip()
assert version.startswith("Python 3.9."), version
print("Isolated SuPreM interpreter:", version)
print("Environment:", VENV)

## 2. Install the verified dependency stack

The installation order below is deliberate and matches the environment that successfully ran the audited demo:

- PyTorch 1.11.0 + CUDA 11.3
- NumPy 1.24.4
- MONAI 0.9.0
- remaining SuPreM runtime packages
- Streamlit 1.49.1 for the Python 3.9 web layer

In [ ]:
if ENV_MARKER.exists():
    print("Verified BodyMaps environment marker found; rechecking packages.")
else:
    run([
        UV, "pip", "install", "--python", str(VENV),
        "torch==1.11.0+cu113",
        "torchvision==0.12.0+cu113",
        "--extra-index-url", "https://download.pytorch.org/whl/cu113",
    ])
    run([UV, "pip", "install", "--python", str(VENV), "numpy==1.24.4"])
    run([UV, "pip", "install", "--python", str(VENV), "monai==0.9.0"])

    support_packages = [
        "numpy==1.24.4",
        "h5py==3.6.0",
        "nibabel",
        "SimpleITK",
        "connected-components-3d",
        "fastremap",
        "tqdm",
        "einops",
        "timm",
        "ml-collections",
        "scipy",
        "pandas",
        "matplotlib",
        "glob2",
        "opencv-python",
        "elasticdeform",
        "streamlit==1.49.1",
    ]
    run([UV, "pip", "install", "--python", str(VENV)] + support_packages)

verify = r'''
import torch, numpy, monai, nibabel, SimpleITK, cc3d, fastremap, streamlit
assert torch.__version__.startswith("1.11.0"), torch.__version__
assert numpy.__version__ == "1.24.4", numpy.__version__
assert monai.__version__ == "0.9.0", monai.__version__
assert streamlit.__version__ == "1.49.1", streamlit.__version__
assert torch.cuda.is_available(), "No PyTorch GPU access"
x = torch.randn(64, 64, device="cuda")
assert (x @ x).shape == (64, 64)
print("ENVIRONMENT PASS")
print({
    "torch": torch.__version__,
    "numpy": numpy.__version__,
    "monai": monai.__version__,
    "streamlit": streamlit.__version__,
    "gpu": torch.cuda.get_device_name(0),
})
'''
run([VENV, "-c", verify])

ENV_MARKER.write_text("BodyMaps final SuPreM environment verified\n")
print("Environment marker:", ENV_MARKER)

## 3. Restore the pinned SuPreM source

The demonstration uses SuPreM commit:

`3e729951c61f729cd2fa6ceefa8e021bcfcd4985`

The notebook checks out that exact commit before importing the model.

In [ ]:
UPSTREAM_COMMIT = "3e729951c61f729cd2fa6ceefa8e021bcfcd4985"

if not (REPO / ".git").exists():
    run(["git", "clone", "https://github.com/MrGiovanni/SuPreM.git", str(REPO)])

run(["git", "-C", str(REPO), "checkout", "--detach", UPSTREAM_COMMIT])

actual_git = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip()
assert actual_git == UPSTREAM_COMMIT, (actual_git, UPSTREAM_COMMIT)

run([VENV, "-c", r'''
import torch, monai, nibabel, SimpleITK, cc3d, fastremap
from model.Universal_model import Universal_model
from dataset.dataloader_test import get_loader
from utils.utils import organ_post_process, NUM_CLASS, ORGAN_NAME_LOW, TEMPLATE
print("SuPreM imports PASS; output classes:", NUM_CLASS)
'''], cwd=SRC, show_output=False)

print("SuPreM source PASS:", actual_git)

## 4. Download and authenticate the official pretrained checkpoint

The checkpoint is **not** stored in this repository. The notebook downloads the author-provided file and verifies its SHA-256 **before** deserialization.

Expected SHA-256:

`c8dc39c86cd8d715e5ef0b5bb4ba4335ba45237127829ea016f75ee8beca0a93`

In [ ]:
CHECKPOINT = SRC / "pretrained_checkpoints" / "supervised_suprem_unet_2100.pth"
CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)

EXPECTED_SHA256 = "c8dc39c86cd8d715e5ef0b5bb4ba4335ba45237127829ea016f75ee8beca0a93"
CHECKPOINT_URL = (
    "https://huggingface.co/MrGiovanni/SuPreM/resolve/main/"
    "supervised_suprem_unet_2100.pth"
)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for part in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(part)
    return h.hexdigest()

if not CHECKPOINT.exists() or sha256(CHECKPOINT) != EXPECTED_SHA256:
    CHECKPOINT.unlink(missing_ok=True)
    temp = CHECKPOINT.with_suffix(".download")
    temp.unlink(missing_ok=True)
    run([
        "curl", "--fail", "--location", "--retry", "3",
        "--connect-timeout", "30",
        "--output", str(temp),
        CHECKPOINT_URL,
    ])
    if sha256(temp) != EXPECTED_SHA256:
        temp.unlink(missing_ok=True)
        raise RuntimeError("Checkpoint SHA-256 mismatch; refusing to use the file.")
    temp.replace(CHECKPOINT)

assert sha256(CHECKPOINT) == EXPECTED_SHA256
print("CHECKPOINT HASH PASS:", EXPECTED_SHA256)
print("Checkpoint size:", round(CHECKPOINT.stat().st_size / 1024**2, 2), "MiB")

## 5. Strict checkpoint audit and audited inference copy

The upstream direct-inference script assigns checkpoint tensors by iteration order. This notebook does **not** use that mapping.

Instead, it:

- verifies the checkpoint hash first;
- strips only recognized `module.` / `model.` prefixes;
- checks model/checkpoint key sets, tensor shapes and dtypes;
- rejects collisions, missing keys and unexpected keys;
- calls `load_state_dict(..., strict=True)`;
- patches a **copy** named `inference_bodymaps_verified.py`;
- leaves upstream `inference.py` unchanged.

PyTorch 1.11 does not provide the newer `weights_only` loading option, so SHA verification is performed before `torch.load`.

In [ ]:
AUDIT_SCRIPT = '''"""Verified SuPreM checkpoint loader for the BodyMaps research demo.

Only the fixed, SHA-verified author checkpoint is deserialized.
"""

import hashlib
import json
from pathlib import Path
from collections.abc import Mapping

EXPECTED_HASH = "c8dc39c86cd8d715e5ef0b5bb4ba4335ba45237127829ea016f75ee8beca0a93"
REPORT_PATH = "/content/bodymaps_reports/checkpoint_audit.json"


def digest(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def normalize(name):
    while name.startswith(("module.", "model.")):
        name = name.split(".", 1)[1]
    return name


def verified_state(model, path):
    import torch

    p = Path(path)

    report = {
        "checkpoint": p.name,
        "sha256": digest(p),
        "expected_sha256": EXPECTED_HASH,
        "audit_pass": False,
        "loader": "strict-name-based",
    }

    try:
        if report["sha256"] != EXPECTED_HASH:
            raise RuntimeError(
                "Checkpoint integrity check FAILED; refusing torch.load"
            )

        blob = torch.load(str(p), map_location="cpu")

        if not isinstance(blob, Mapping) or not isinstance(blob.get("net"), Mapping):
            raise RuntimeError(
                "Expected checkpoint dictionary containing state_dict in 'net'"
            )

        model_state = model.state_dict()
        checkpoint_state = blob["net"]

        normalized = {}
        collisions = []

        for name, tensor in checkpoint_state.items():
            key = normalize(name)

            if key in normalized:
                collisions.append(key)

            normalized[key] = tensor

        model_keys = set(model_state)
        checkpoint_keys = set(normalized)

        missing = sorted(model_keys - checkpoint_keys)
        unexpected = sorted(checkpoint_keys - model_keys)

        wrong_shape = []
        wrong_dtype = []

        for key in sorted(model_keys & checkpoint_keys):
            model_tensor = model_state[key]
            checkpoint_tensor = normalized[key]

            if (
                not isinstance(checkpoint_tensor, torch.Tensor)
                or tuple(model_tensor.shape) != tuple(checkpoint_tensor.shape)
            ):
                wrong_shape.append(key)

            elif model_tensor.dtype != checkpoint_tensor.dtype:
                wrong_dtype.append(key)

        report.update(
            model_tensor_count=len(model_keys),
            checkpoint_tensor_count=len(checkpoint_keys),
            missing=missing,
            unexpected=unexpected,
            duplicate_normalized_keys=collisions,
            wrong_shape=wrong_shape,
            wrong_dtype=wrong_dtype,
        )

        if collisions or missing or unexpected or wrong_shape or wrong_dtype:
            raise RuntimeError(
                "Checkpoint semantic audit FAILED; inspect checkpoint_audit.json"
            )

        model.load_state_dict(normalized, strict=True)

        report["audit_pass"] = True

        return normalized

    except Exception as exc:
        report["error"] = str(exc)
        raise

    finally:
        Path(REPORT_PATH).parent.mkdir(parents=True, exist_ok=True)
        Path(REPORT_PATH).write_text(
            json.dumps(report, indent=2),
            encoding="utf-8",
        )
'''


AUDIT_FILE = SRC / "bodymaps_checkpoint.py"
AUDIT_FILE.write_text(AUDIT_SCRIPT, encoding="utf-8")

# Sanity-check the generated Python file before running it.
compile(
    AUDIT_FILE.read_text(encoding="utf-8"),
    str(AUDIT_FILE),
    "exec",
)

print("Checkpoint audit helper syntax: PASS")


upstream_file = SRC / "inference.py"
source = upstream_file.read_text(encoding="utf-8")

start = source.find("        #Load pre-trained weights")
stop = source.find("\n    if args.customize:", start)

if start < 0 or stop < 0 or source.count("        #Load pre-trained weights") != 1:
    raise RuntimeError(
        "SuPreM upstream loader block changed; review before patching."
    )

replacement = (
    "        # Verified name-based checkpoint loading; never positional.\n"
    "        from bodymaps_checkpoint import verified_state\n"
    "        store_dict = verified_state(model, args.checkpoint)\n"
)

VERIFIED_INFERENCE = SRC / "inference_bodymaps_verified.py"
VERIFIED_INFERENCE.write_text(
    source[:start] + replacement + source[stop:],
    encoding="utf-8",
)

verified_source = VERIFIED_INFERENCE.read_text(encoding="utf-8")

assert (
    "load_dict_value = [value for key, value in load_dict.items()]"
    not in verified_source
)

compile(
    verified_source,
    str(VERIFIED_INFERENCE),
    "exec",
)

print("Created audited inference copy:", VERIFIED_INFERENCE.name)
print("Original upstream inference.py remains unchanged.")
print("Audited inference script syntax: PASS")


AUDIT_RUNNER = """
from pathlib import Path

from model.Universal_model import Universal_model
from utils.utils import NUM_CLASS
from bodymaps_checkpoint import verified_state

checkpoint = Path(
    "pretrained_checkpoints/supervised_suprem_unet_2100.pth"
)

model = Universal_model(
    img_size=(96, 96, 96),
    in_channels=1,
    out_channels=NUM_CLASS,
    backbone="unet",
    encoding="word_embedding",
)

verified_state(model, checkpoint)

print(
    "STRICT CHECKPOINT AUDIT PASS; model tensors:",
    len(model.state_dict()),
)
"""

run(
    [VENV, "-c", AUDIT_RUNNER],
    cwd=SRC,
    show_output=False,
)

audit = json.loads(
    (REPORT / "checkpoint_audit.json").read_text(encoding="utf-8")
)

assert audit["audit_pass"] is True

print(json.dumps(audit, indent=2))

## 6. Fetch the reviewed web application from the public repository

To avoid duplicating a very large application source cell, the notebook downloads the exact reviewed `app.py` and `inference_backend.py` from a pinned GitHub commit.

Pinned web-app commit:

`7a4471810458eaf73c5687f8a2aaa0d103ff148c`

The backend preserves the important integration fix: `SUPREM_PYTHON` is **not** resolved to the base interpreter. It must remain `/content/suprem_env/bin/python`, where PyTorch and MONAI are installed.

In [ ]:
PROJECT_COMMIT = "7a4471810458eaf73c5687f8a2aaa0d103ff148c"
RAW_BASE = (
    "https://raw.githubusercontent.com/sojore/"
    f"BodyMaps-SuPreM-CT-AI-Lab/{PROJECT_COMMIT}"
)

for filename in ("app.py", "inference_backend.py"):
    url = f"{RAW_BASE}/{filename}"
    target = WEBAPP / filename
    urllib.request.urlretrieve(url, target)
    if not target.is_file() or target.stat().st_size == 0:
        raise RuntimeError(f"Failed to retrieve {filename}")
    print("Fetched:", filename, "bytes:", target.stat().st_size)

backend_text = (WEBAPP / "inference_backend.py").read_text()
assert "python = Path(os.environ['SUPREM_PYTHON'])" in backend_text
assert "python = Path(os.environ['SUPREM_PYTHON']).resolve()" not in backend_text

run([VENV, "-m", "py_compile", WEBAPP / "app.py", WEBAPP / "inference_backend.py"])
print("WEB APP SOURCE PASS — pinned commit:", PROJECT_COMMIT)

## 7. Launch Streamlit on the Colab GPU host

The Streamlit process is private to the Colab VM. Internal subprocess paths and logs are not exposed in browser error messages.

The notebook first verifies that Streamlit answers locally on port `8501`.

In [ ]:
PORT = 8501
streamlit_pid_file = ROOT / "bodymaps_streamlit.pid"

if streamlit_pid_file.exists():
    try:
        old_pid = int(streamlit_pid_file.read_text().strip())
        os.kill(old_pid, 15)
        time.sleep(1)
    except Exception:
        pass
    streamlit_pid_file.unlink(missing_ok=True)

streamlit_log = REPORT / "streamlit_web.log"
streamlit_handle = open(streamlit_log, "w")

streamlit_proc = subprocess.Popen(
    [
        str(VENV), "-m", "streamlit", "run", str(WEBAPP / "app.py"),
        "--server.port", str(PORT),
        "--server.address", "0.0.0.0",
        "--server.headless", "true",
        "--server.fileWatcherType", "none",
        "--browser.gatherUsageStats", "false",
    ],
    cwd=str(WEBAPP),
    env={
        **RUN_ENV,
        "SUPREM_REPO": str(REPO),
        "SUPREM_CHECKPOINT": str(CHECKPOINT),
        "SUPREM_PYTHON": str(VENV),
        "BODYMAPS_REPORT_DIR": str(REPORT),
    },
    stdout=streamlit_handle,
    stderr=subprocess.STDOUT,
)

streamlit_pid_file.write_text(str(streamlit_proc.pid))

for _ in range(60):
    if streamlit_proc.poll() is not None:
        streamlit_handle.flush()
        raise RuntimeError("Streamlit exited early; inspect /content/bodymaps_reports/streamlit_web.log")
    with socket.socket() as s:
        if s.connect_ex(("127.0.0.1", PORT)) == 0:
            break
    time.sleep(0.5)
else:
    raise RuntimeError("Streamlit did not open port 8501.")

status = urllib.request.urlopen(f"http://127.0.0.1:{PORT}", timeout=10).status
assert status == 200, status

print("STREAMLIT SERVER PASS")
print("PID:", streamlit_proc.pid)
print("Local HTTP status:", status)

## 8. Open the temporary browser demo

Colab's built-in proxy was unreliable during the verified run, so the final notebook uses a **Cloudflare Quick Tunnel** for the development demonstration.

The tunnel URL is temporary. Keep the Colab runtime open while using it.

**Use only anonymized research/test CTs. Do not share the temporary URL publicly.**

In [ ]:
CLOUDFLARED = ROOT / "cloudflared"
CLOUDFLARED_URL = (
    "https://github.com/cloudflare/cloudflared/releases/latest/download/"
    "cloudflared-linux-amd64"
)

if not CLOUDFLARED.exists():
    urllib.request.urlretrieve(CLOUDFLARED_URL, CLOUDFLARED)
    os.chmod(CLOUDFLARED, 0o755)

tunnel_pid_file = ROOT / "bodymaps_cloudflared.pid"
if tunnel_pid_file.exists():
    try:
        old_pid = int(tunnel_pid_file.read_text().strip())
        os.kill(old_pid, 15)
        time.sleep(1)
    except Exception:
        pass
    tunnel_pid_file.unlink(missing_ok=True)

tunnel_log = REPORT / "cloudflared.log"
tunnel_handle = open(tunnel_log, "w")

tunnel_proc = subprocess.Popen(
    [
        str(CLOUDFLARED),
        "tunnel",
        "--no-autoupdate",
        "--url",
        f"http://127.0.0.1:{PORT}",
    ],
    stdout=tunnel_handle,
    stderr=subprocess.STDOUT,
)

tunnel_pid_file.write_text(str(tunnel_proc.pid))

public_url = None
pattern = re.compile(r"https://[A-Za-z0-9-]+\.trycloudflare\.com")

for _ in range(45):
    time.sleep(1)
    tunnel_handle.flush()
    if tunnel_log.exists():
        match = pattern.search(tunnel_log.read_text(errors="replace"))
        if match:
            public_url = match.group(0)
            break
    if tunnel_proc.poll() is not None:
        break

if not public_url:
    print(tunnel_log.read_text(errors="replace")[-12000:])
    raise RuntimeError("Cloudflare Quick Tunnel URL was not created.")

print("BODYMAPS WEB APP")
print(public_url)
print("\nOpen the URL above in a new browser tab.")

## 9. Browser test

In the opened **BodyMaps | CT AI Lab** page:

1. leave **Run SuPreM on uploaded CT (GPU host)** selected;
2. upload an anonymized BodyMaps case ZIP containing `ct.nii.gz` (reference masks are optional);
3. click **Run pretrained SuPreM inference**;
4. require the success message **“Inference completed: 9 organ masks generated.”**;
5. inspect the three-plane prediction overlay;
6. if reference masks are present, inspect error analysis and Dice/IoU;
7. optionally download the generated prediction ZIP and evaluation CSV.

A successful page load alone is **not** sufficient. The browser must trigger fresh SuPreM inference and return all nine masks.

### Verified functional results

The workflow was successfully exercised on three BodyMaps/AbdomenAtlas cases:

| Case | CT shape | Generated masks | Mean Dice | Mean IoU |
|---|---:|---:|---:|---:|
| `BDMAP_00000338` | `(502, 348, 71)` | 9 | 0.9252 | 0.8639 |
| `BDMAP_00000015` | `(502, 342, 33)` | 9 | 0.8930 | 0.8301 |
| `BDMAP_00000018` | `(238, 178, 170)` | 9 | 0.8882 | 0.8172 |

These values are **functional demonstration results**, not an independent held-out benchmark. Training/pretraining overlap for these cases has not been established.

## 10. Troubleshooting only if browser inference fails

The web UI intentionally hides internal filesystem paths. If an inference attempt fails, inspect the most recent private server-side inference log with the cell below.

Do not run this cell unless there is an actual failure.

In [ ]:
from pathlib import Path

logs = sorted(
    REPORT.glob("web_inference_*.log"),
    key=lambda p: p.stat().st_mtime,
)

if not logs:
    print("No browser inference logs exist yet.")
else:
    latest = logs[-1]
    print("Latest private inference log:", latest.name)
    print("=" * 80)
    print(latest.read_text(errors="replace")[-15000:])